# AIC2026 — Hybrid OCR test trên Google Colab

Notebook này thay pipeline **EasyOCR/CRAFT + VietOCR** bằng pipeline thử nghiệm:

```text
PP-OCRv6 text detector → crop phối cảnh + padding dấu → VietOCR recognizer
                     → temporal consensus trên các keyframe lân cận
```

Đây là notebook **test nhỏ**, chưa chạy toàn bộ dataset. Mục tiêu là nhìn bằng mắt:

1. detector có bắt đúng chữ nhỏ trên keyframe hay không;
2. VietOCR có đọc được tiếng Việt có dấu sau khi crop hay không;
3. kết quả có ổn định giữa các keyframe gần nhau hay không;
4. tốc độ thực tế trên Colab T4/L4.

Khác bản cũ:

- không dùng EasyOCR làm detector;
- không đưa cả frame vào VietOCR;
- crop theo polygon và thêm padding theo chiều cao box để không cắt dấu;
- dùng `vgg_transformer` với batch vừa phải để ưu tiên chất lượng trên GPU 16GB;
- lấy các frame lân cận quanh nhiều vị trí trong video để thử temporal voting;
- giữ `frame_idx`, `pts_time`, `bbox`, confidence và xuất `ocr_index.jsonl`.

VLM không chạy trong notebook này. Với cuộc thi, VLM nên để ở bước verifier trên top candidate,
không chạy trên toàn bộ index.
> GPU note: Paddle detector và VietOCR đều chạy trên GPU. Do GPU wheel của Paddle và Torch có thể yêu cầu cuDNN khác phiên bản, notebook tắt riêng Torch cuDNN cho VietOCR; model vẫn chạy bằng CUDA, còn Paddle detector vẫn dùng GPU.
- cấu hình cân bằng: detector medium, crop padding 50%, upscale tối thiểu 64px và không nhân đôi crop khi chạy full 30k ảnh.


In [ ]:
# Cài đặt phải chạy trước mọi import paddle/torch/PIL.
# Cell này tự chọn bản GPU hoặc CPU; không gọi restart kernel.
import shutil
import subprocess
import sys

def pip_install(*args, check=True):
    command = [sys.executable, '-m', 'pip', 'install', '-q', *args]
    print('>>', ' '.join(command))
    return subprocess.run(command, check=check)

def has_visible_nvidia_gpu():
    executable = shutil.which('nvidia-smi')
    if executable is None:
        return False
    return subprocess.run([executable], stdout=subprocess.DEVNULL,
                          stderr=subprocess.DEVNULL).returncode == 0

GPU_AVAILABLE = has_visible_nvidia_gpu()
# Both Paddle detector and VietOCR must use GPU for the full dataset.
# Torch cuDNN is disabled separately below to avoid the Paddle/Torch cuDNN clash.
USE_PADDLE_GPU = True
PADDLE_GPU_ENABLED = GPU_AVAILABLE and USE_PADDLE_GPU
# Both Paddle detector and VietOCR must use GPU for the full dataset.
# Torch cuDNN is disabled separately below to avoid the Paddle/Torch cuDNN clash.
print('NVIDIA GPU visible:', GPU_AVAILABLE, '| Paddle GPU enabled:', PADDLE_GPU_ENABLED)
if not GPU_AVAILABLE:
    print('Không thấy libcuda/driver -> cài Paddle CPU. Nếu muốn dùng GPU, hãy bật GPU runtime trước.')

# Xóa bản Paddle sai nếu cell này từng được chạy trong cùng session.
# Trong Save & Version, cell này nằm đầu notebook nên không cần restart.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q',
                'paddlepaddle', 'paddlepaddle-gpu'], check=False)

if PADDLE_GPU_ENABLED:
    gpu_result = pip_install(
        'paddlepaddle-gpu',
        '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu126/',
        check=False,
    )
    if gpu_result.returncode != 0:
        print('cu126 không cài được, thử wheel cu118...')
        pip_install('paddlepaddle-gpu',
                    '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu118/')
else:
    pip_install('paddlepaddle',
                '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cpu/')

# Nếu notebook trước đó đã thử import Paddle và bị lỗi, xóa module dở dang
# khỏi kernel hiện tại để cell kế tiếp import được bản vừa cài.
for module_name in list(sys.modules):
    if (module_name == 'paddle' or module_name.startswith('paddle.')
            or module_name == 'torch' or module_name.startswith('torch.')):
        del sys.modules[module_name]

# Paddle GPU can downgrade NCCL and break Torch's libtorch_cuda.so.
if PADDLE_GPU_ENABLED:
    pip_install('--upgrade', '--no-deps', 'nvidia-nccl-cu12>=2.27', check=False)

pip_install('paddleocr>=3.3,<4')

# VietOCR pin Pillow cũ; --no-deps tránh hạ Pillow/Torch của runtime.
pip_install('--no-deps', 'vietocr')
pip_install('einops', 'gdown')

# Kiểm tra ngay trong subprocess độc lập để biết wheel đã cài đúng.
check = subprocess.run(
    [sys.executable, '-c',
     'import torch; print("Torch", torch.__version__); import paddle; print("Paddle", paddle.__version__, "CUDA", paddle.device.is_compiled_with_cuda())'],
    text=True, capture_output=True)
print(check.stdout)
if check.returncode != 0:
    print(check.stderr[-4000:])
    raise RuntimeError('Paddle import thất bại. Kiểm tra GPU runtime hoặc dùng CPU runtime mới.')

# Chỉ chạy nvidia-smi sau khi cài đặt nếu muốn xem lại GPU.
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================ CẤU HÌNH ============================
from pathlib import Path
import csv
import re
import numpy as np

# Sửa dòng này nếu Dataset_Directory của bạn nằm ở vị trí khác trong Drive.
DATASET_DIRECTORY = Path('/content/drive/MyDrive/AI Challenge/Dataset_Directory')

# None = tự tìm mọi thư mục Keyframes_*. Có thể thay bằng một vài folder để test nhanh.
TARGET_FOLDERS = None
TARGET_VIDEO_IDS = []       # ví dụ: ['L21_V001']; [] = không lọc theo video

SAMPLE_VIDEOS = 3           # số video test
CENTERS_PER_VIDEO = 6       # số vị trí rải đều trong mỗi video
TEMPORAL_RADIUS = 1         # lấy thêm frame trước/sau mỗi vị trí; 1 = 3 frame/cụm

DETECTOR_MODEL = 'PP-OCRv6_medium_det'
VIETOCR_CONFIG_NAME = 'vgg_transformer'  # cân bằng accuracy/tốc độ cho GPU 16GB
DETECTOR_SCORE_FLOOR = 0.22
RECOGNIZER_SCORE_FLOOR = 0.25
MAX_BOXES_PER_IMAGE = 24
RECOGNITION_BATCH = 16
USE_ENHANCED_VARIANT = False  # chỉ bật khi test ảnh khó; tránh nhân đôi thời gian full run

OUTPUT_DIRECTORY = Path('/content/aic_ocr_hybrid_test')
MODEL_DIRECTORY = Path('/content/aic_ocr_hybrid_models')
PREVIEW_COUNT = 8
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}
VIDEO_ID_PATTERN = re.compile(r'^L\d{2}_V\d{3}$')

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
MODEL_DIRECTORY.mkdir(parents=True, exist_ok=True)

def find_map_directory(root):
    candidates = [
        root / 'map-keyframes-aic25-b1' / 'map-keyframes',
        root / 'map-keyframes',
    ]
    candidates += sorted(root.glob('*/map-keyframes'))
    candidates += sorted(root.glob('map-keyframes*/map-keyframes*'))
    return next((p for p in candidates if p.is_dir()), None)

def find_video_dirs(root):
    base = root / 'keyframes' if (root / 'keyframes').is_dir() else root
    return [p for p in sorted(base.iterdir())
            if p.is_dir() and VIDEO_ID_PATTERN.match(p.name)]

def list_images(video_dir):
    def order(path):
        return int(path.stem) if path.stem.isdigit() else 10**12
    return sorted((p for p in video_dir.iterdir()
                   if p.suffix.lower() in IMAGE_EXTENSIONS),
                  key=lambda p: (order(p), p.name))

if not DATASET_DIRECTORY.is_dir():
    print('Không thấy dataset:', DATASET_DIRECTORY)
    print('Các thư mục trong MyDrive/AI Challenge:')
    parent = DATASET_DIRECTORY.parent
    if parent.is_dir():
        for p in sorted(parent.iterdir()):
            print(' ', p)
    raise FileNotFoundError('Sửa DATASET_DIRECTORY rồi chạy lại cell này.')

MAP_KEYFRAMES_DIRECTORY = find_map_directory(DATASET_DIRECTORY)
if MAP_KEYFRAMES_DIRECTORY is None:
    print('CẢNH BÁO: không thấy map-keyframes; frame_idx sẽ để None.')
else:
    print('Map:', MAP_KEYFRAMES_DIRECTORY)

if TARGET_FOLDERS is None:
    keyframe_roots = sorted(p for p in DATASET_DIRECTORY.iterdir()
                            if p.is_dir() and p.name.startswith('Keyframes'))
else:
    keyframe_roots = [DATASET_DIRECTORY / name for name in TARGET_FOLDERS
                      if (DATASET_DIRECTORY / name).is_dir()]

all_video_dirs = sorted((v for root in keyframe_roots for v in find_video_dirs(root)),
                        key=lambda p: p.name)
if TARGET_VIDEO_IDS:
    all_video_dirs = [p for p in all_video_dirs if p.name in set(TARGET_VIDEO_IDS)]
if not all_video_dirs:
    raise FileNotFoundError('Không tìm thấy video Lxx_Vxxx trong các thư mục Keyframes.')

def choose_evenly(items, count):
    if len(items) <= count:
        return list(items)
    indexes = np.linspace(0, len(items) - 1, count).round().astype(int)
    return [items[int(i)] for i in indexes]

def make_temporal_sample(images):
    if not images:
        return []
    centers = np.linspace(0, len(images) - 1,
                          min(CENTERS_PER_VIDEO, len(images))).round().astype(int)
    indexes = set()
    for center in centers:
        for offset in range(-TEMPORAL_RADIUS, TEMPORAL_RADIUS + 1):
            indexes.add(max(0, min(len(images) - 1, int(center) + offset)))
    return [images[i] for i in sorted(indexes)]

sampled_video_dirs = choose_evenly(all_video_dirs, SAMPLE_VIDEOS)
sampled_groups = [(video_dir, make_temporal_sample(list_images(video_dir)))
                  for video_dir in sampled_video_dirs]
sampled_groups = [(v, paths) for v, paths in sampled_groups if paths]
sampled_paths = [path for _, paths in sampled_groups for path in paths]

print(f'{len(all_video_dirs)} video khả dụng; {len(sampled_paths)} ảnh test.')
for video_dir, paths in sampled_groups:
    print(f'  {video_dir.name}: {len(paths)} keyframe, gồm các frame lân cận')

In [ ]:
# ============================ NẠP MODEL ============================
import time
import torch
# Paddle GPU currently ships a different cuDNN than the Colab Torch wheel.
# Keep VietOCR on CUDA, but use Torch native convolutions to avoid cuDNN loading.
torch.backends.cudnn.enabled = False
import paddle
from PIL import Image, ImageOps
from paddleocr import TextDetection
from vietocr.tool.config import Cfg
from vietocr.tool.predictor import Predictor

paddle_gpu = bool(PADDLE_GPU_ENABLED and paddle.device.is_compiled_with_cuda()
                 and paddle.device.cuda.device_count() > 0)
PADDLE_DEVICE = 'gpu:0' if paddle_gpu else 'cpu'
TORCH_DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('Paddle:', PADDLE_DEVICE, '| Torch:', TORCH_DEVICE)
if PADDLE_DEVICE != 'gpu:0' or TORCH_DEVICE != 'cuda:0':
    raise RuntimeError('Notebook này cần Colab/Kaggle bật NVIDIA GPU trước khi chạy.')

t0 = time.perf_counter()
detector = TextDetection(model_name=DETECTOR_MODEL, device=PADDLE_DEVICE)
print(f'Detector loaded: {DETECTOR_MODEL} in {time.perf_counter() - t0:.1f}s')

# VietOCR sẽ tự tải pretrained weight ở lần đầu.
vietocr_cfg = Cfg.load_config_from_name(VIETOCR_CONFIG_NAME)
vietocr_cfg['device'] = TORCH_DEVICE
vietocr_cfg['cnn']['pretrained'] = False
vietocr_cfg.setdefault('predictor', {})['beamsearch'] = False
recognizer = Predictor(vietocr_cfg)
print(f'Recognizer loaded: {VIETOCR_CONFIG_NAME}')

In [ ]:
# ============================ DETECTION + CROP ============================
import json
import cv2
import math
import unicodedata
from difflib import SequenceMatcher
from PIL import ImageEnhance

VIETNAMESE_PROBES = set(
    'ăắằẳẵặâấầẩẫậđêếềểễệôốồổỗộơớờởỡợưứừửữự'
    'áàảãạéèẻẽẹíìỉĩịóòỏõọúùủũụýỳỷỹỵ'
    'ĂẮẰẲẴẶÂẤẦẨẪẬĐÊẾỀỂỄỆÔỐỒỔỖỘƠỚỜỞỠỢƯỨỪỬỮỰ'
    'ÁÀẢÃẠÉÈẺẼẸÍÌỈĨỊÓÒỎÕỌÚÙỦŨỤÝỲỶỸỴ'
)

def load_rgb(path):
    return ImageOps.exif_transpose(Image.open(path)).convert('RGB')

def unwrap_result(value):
    """Chuẩn hóa Result object/dict của PaddleOCR 3.x về dict."""
    if isinstance(value, dict):
        nested = value.get('res')
        return nested if isinstance(nested, dict) else value
    for attribute in ('json', 'to_dict', 'res'):
        if not hasattr(value, attribute):
            continue
        candidate = getattr(value, attribute)
        try:
            candidate = candidate() if callable(candidate) else candidate
        except TypeError:
            continue
        if isinstance(candidate, str):
            try:
                candidate = json.loads(candidate)
            except json.JSONDecodeError:
                continue
        if isinstance(candidate, dict):
            nested = candidate.get('res')
            return nested if isinstance(nested, dict) else candidate
    return {}

def polygon_to_array(polygon):
    points = np.asarray(polygon, dtype=np.float32).reshape(-1, 2)
    if len(points) < 4:
        return None
    return points[:4]

def xyxy_from_polygon(points):
    points = np.asarray(points)
    return [int(points[:, 0].min()), int(points[:, 1].min()),
            int(points[:, 0].max()), int(points[:, 1].max())]

def detect_polygons(path):
    outputs = list(detector.predict(str(path), batch_size=1))
    if not outputs:
        return []
    data = unwrap_result(outputs[0])
    polygons = data.get('dt_polys')
    if polygons is None or len(polygons) == 0:
        polygons = data.get('rec_polys')
    if polygons is None:
        polygons = []
    scores = data.get('dt_scores')
    if scores is None or len(scores) == 0:
        scores = [1.0] * len(polygons)
    rows = []
    for polygon, score in zip(polygons, scores):
        points = polygon_to_array(polygon)
        if points is None:
            continue
        score = float(score)
        x0, y0, x1, y1 = xyxy_from_polygon(points)
        if score >= DETECTOR_SCORE_FLOOR and x1 > x0 and y1 > y0:
            rows.append({'polygon': points, 'det_conf': score,
                         'bbox': [x0, y0, x1, y1]})
    rows.sort(key=lambda row: (-row['det_conf'], row['bbox'][1], row['bbox'][0]))
    return rows[:MAX_BOXES_PER_IMAGE]

def order_quad(points):
    points = np.asarray(points, dtype=np.float32)
    total = points.sum(axis=1)
    diff = np.diff(points, axis=1).ravel()
    return np.array([points[np.argmin(total)], points[np.argmin(diff)],
                     points[np.argmax(total)], points[np.argmax(diff)]], dtype=np.float32)

def rectify_crop(image, polygon):
    """Warp polygon thành dòng ngang, sau đó thêm padding dọc để giữ dấu."""
    source = order_quad(polygon)
    width = int(max(np.linalg.norm(source[2] - source[3]),
                    np.linalg.norm(source[1] - source[0])))
    height = int(max(np.linalg.norm(source[1] - source[2]),
                     np.linalg.norm(source[0] - source[3])))
    if width < 6 or height < 4:
        return None
    target = np.array([[0, 0], [width - 1, 0],
                       [width - 1, height - 1], [0, height - 1]], dtype=np.float32)
    matrix = cv2.getPerspectiveTransform(source, target)
    rgb = np.asarray(image)
    warped = cv2.warpPerspective(rgb, matrix, (width, height),
                                 borderMode=cv2.BORDER_REPLICATE)

    # Padding động quan trọng với tiếng Việt: detector ôm sát có thể cắt mũ/dấu.
    pad_x = max(2, int(width * 0.04))
    pad_y = max(3, int(height * 0.50))
    warped = cv2.copyMakeBorder(warped, pad_y, pad_y, pad_x, pad_x,
                                cv2.BORDER_REPLICATE)
    scale = max(1.0, 64.0 / max(1, warped.shape[0]))
    if scale > 1.0:
        warped = cv2.resize(warped, None, fx=scale, fy=scale,
                            interpolation=cv2.INTER_CUBIC)
    return Image.fromarray(warped).convert('RGB')

def crop_variants(crop):
    """Hai variant nhẹ; variant sharpen chỉ dùng để cứu crop mờ."""
    rgb = np.asarray(crop)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    enhanced = cv2.cvtColor(clahe, cv2.COLOR_GRAY2RGB)
    enhanced = Image.fromarray(enhanced)
    enhanced = ImageEnhance.Sharpness(enhanced).enhance(1.35)
    return [crop, enhanced] if USE_ENHANCED_VARIANT else [crop]

def clean_text(text):
    return unicodedata.normalize('NFC', ' '.join(str(text or '').split())).strip()

def is_meaningful(text):
    if len(text) < 2:
        return False
    return any(ch.isalnum() for ch in text)

def parse_recognition(item):
    if isinstance(item, (tuple, list)):
        text = item[0] if item else ''
        score = item[1] if len(item) > 1 else 0.0
    else:
        text, score = item, 0.0
    try:
        score = float(score)
    except (TypeError, ValueError):
        score = 0.0
    return clean_text(text), score

In [ ]:
# ============================ RECOGNITION ============================
def predict_vietocr(crops):
    if not crops:
        return []
    outputs = []
    for start in range(0, len(crops), RECOGNITION_BATCH):
        batch = crops[start:start + RECOGNITION_BATCH]
        try:
            with torch.no_grad():
                pred = recognizer.predict_batch(batch, return_prob=True)
        except TypeError:
            with torch.no_grad():
                pred = recognizer.predict_batch(batch)
            pred = [(value, 0.0) for value in pred]
        outputs.extend(parse_recognition(item) for item in pred)
    return outputs

def ocr_one_image(path):
    image = load_rgb(path)
    detections = detect_polygons(path)
    crops, crop_meta = [], []
    for index, row in enumerate(detections):
        crop = rectify_crop(image, row['polygon'])
        if crop is None:
            continue
        for variant_index, variant in enumerate(crop_variants(crop)):
            crops.append(variant)
            crop_meta.append((index, variant_index, row, row['polygon'].tolist()))

    predictions = predict_vietocr(crops)
    best = {}
    for meta, (text, rec_conf) in zip(crop_meta, predictions):
        index, variant_index, row, polygon = meta
        candidate = {'text': text, 'rec_conf': rec_conf,
                     'det_conf': row['det_conf'], 'bbox': row['bbox'],
                     'polygon': polygon, 'variant': variant_index}
        if index not in best or rec_conf > best[index]['rec_conf']:
            best[index] = candidate

    boxes = [item for item in best.values()
             if item['text'] and is_meaningful(item['text'])]
    boxes.sort(key=lambda item: (item['bbox'][1], item['bbox'][0]))
    kept = [item for item in boxes if item['rec_conf'] >= RECOGNIZER_SCORE_FLOOR]
    text = '\n'.join(item['text'] for item in kept)
    gray = cv2.cvtColor(np.asarray(image), cv2.COLOR_RGB2GRAY)
    sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    return {'path': str(path), 'keyframe': int(path.stem) if path.stem.isdigit() else None,
            'text': text, 'boxes': boxes, 'sharpness': sharpness,
            'has_vietnamese_diacritic': any(set(item['text']) & VIETNAMESE_PROBES
                                            for item in boxes)}

In [ ]:
# ============================ SMOKE TEST ============================
import time
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as PlotPolygon

if not sampled_paths:
    raise RuntimeError('Không có ảnh mẫu.')

test_paths = sampled_paths
started = time.perf_counter()
RESULTS = []
for index, path in enumerate(test_paths, 1):
    try:
        result = ocr_one_image(path)
        result['video_id'] = path.parent.name
        RESULTS.append(result)
        print(f'[{index:03d}/{len(test_paths)}] {path.parent.name}/{path.name}: '
              f'{len(result["boxes"])} box | {result["text"].replace(chr(10), " | ")[:160]}')
    except Exception as exc:
        print('LỖI', path, type(exc).__name__, exc)

elapsed = time.perf_counter() - started
print(f'\nTốc độ: {len(RESULTS) / max(elapsed, 1e-9):.2f} ảnh/s '
      f'({elapsed / max(len(RESULTS), 1) * 1000:.0f} ms/ảnh)')
print('Ảnh có text:', sum(bool(row['text']) for row in RESULTS), '/', len(RESULTS))
print('Ảnh có dấu tiếng Việt:', sum(row['has_vietnamese_diacritic'] for row in RESULTS))

def show_preview(result):
    image = load_rgb(result['path'])
    figure, axes = plt.subplots(1, 2, figsize=(18, 7),
                                gridspec_kw={'width_ratios': [1.7, 1]})
    axes[0].imshow(image)
    axes[0].set_title(f'{result["video_id"]}/{Path(result["path"]).name}')
    axes[0].axis('off')
    for item in result['boxes']:
        polygon = np.asarray(item['polygon'])
        color = 'lime' if item['rec_conf'] >= RECOGNIZER_SCORE_FLOOR else 'tomato'
        axes[0].add_patch(PlotPolygon(polygon, closed=True, fill=False,
                                       edgecolor=color, linewidth=2))
        x, y = polygon[:, 0].min(), polygon[:, 1].min()
        axes[0].text(x, max(0, y - 3),
                     f'{item["text"][:30]} ({item["rec_conf"]:.2f})',
                     color='black', fontsize=8,
                     bbox={'facecolor': color, 'alpha': 0.8, 'pad': 2})
    body = '\n'.join(f'[{item["rec_conf"]:.2f}] {item["text"]}'
                     for item in result['boxes']) or '(không đọc được text)'
    axes[1].text(0, 1, body[:1800], va='top', fontsize=11, wrap=True)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

nonempty = [row for row in RESULTS if row['boxes']]
for result in (nonempty[:PREVIEW_COUNT] or RESULTS[:PREVIEW_COUNT]):
    show_preview(result)

In [ ]:
# ============================ XEM CROP ============================
# Cell này giúp phân biệt detector sai với recognizer sai.
from IPython.display import display

def show_crops(result, limit=12):
    image = load_rgb(result['path'])
    shown = 0
    for item in result['boxes']:
        crop = rectify_crop(image, np.asarray(item['polygon'], dtype=np.float32))
        if crop is None:
            continue
        print(f'{item["text"]!r} | det={item["det_conf"]:.2f} '
              f'rec={item["rec_conf"]:.2f} | bbox={item["bbox"]}')
        display(crop.resize((min(1000, crop.width),
                             max(32, int(crop.height * min(1000, crop.width) / crop.width)))))
        shown += 1
        if shown >= limit:
            break

if RESULTS:
    candidate = next((row for row in RESULTS if row['boxes']), RESULTS[0])
    print('Crops của:', candidate['path'])
    show_crops(candidate)

In [ ]:
# ============================ TEMPORAL CONSENSUS ============================
def similarity(left, right):
    left = clean_text(left).casefold()
    right = clean_text(right).casefold()
    return SequenceMatcher(None, left, right).ratio()

def temporal_consensus(rows, similarity_floor=0.78):
    """Gom các dòng gần giống xuất hiện trên nhiều keyframe cùng video."""
    clusters = []
    for row in sorted(rows, key=lambda item: item['keyframe'] if item['keyframe'] is not None else 10**12):
        for box in row['boxes']:
            text = box['text']
            target = next((cluster for cluster in clusters
                           if similarity(cluster['representative'], text) >= similarity_floor), None)
            if target is None:
                clusters.append({'representative': text, 'members': [box],
                                 'frames': {row['keyframe']}})
            else:
                previous_best = max(item['rec_conf'] for item in target['members'])
                target['members'].append(box)
                target['frames'].add(row['keyframe'])
                if box['rec_conf'] > previous_best:
                    target['representative'] = text

    consensus = []
    for cluster in clusters:
        best = max(cluster['members'], key=lambda item: item['rec_conf'])
        consensus.append({'text': cluster['representative'],
                          'support': len(cluster['frames']),
                          'best_conf': best['rec_conf'],
                          'frames': sorted(f for f in cluster['frames'] if f is not None)})
    return sorted(consensus, key=lambda item: (-item['support'], -item['best_conf']))

CONSENSUS_BY_VIDEO = {}
for video_id in sorted({row['video_id'] for row in RESULTS}):
    rows = [row for row in RESULTS if row['video_id'] == video_id]
    CONSENSUS_BY_VIDEO[video_id] = temporal_consensus(rows)
    print(f'\n{video_id}')
    for item in CONSENSUS_BY_VIDEO[video_id][:20]:
        print(f'  support={item["support"]} conf={item["best_conf"]:.2f} '
              f'frames={item["frames"][:8]} | {item["text"]}')

In [ ]:
# ============================ EXPORT ============================
def load_keyframe_map(video_id):
    if MAP_KEYFRAMES_DIRECTORY is None:
        return {}
    path = MAP_KEYFRAMES_DIRECTORY / f'{video_id}.csv'
    if not path.is_file():
        return {}
    mapping = {}
    with path.open(encoding='utf-8-sig', newline='') as handle:
        for row in csv.DictReader(handle):
            try:
                mapping[int(row['n'])] = {
                    'frame_idx': int(row['frame_idx']),
                    'pts_time': float(row['pts_time']),
                    'fps': float(row['fps']),
                }
            except (KeyError, TypeError, ValueError):
                continue
    return mapping

index_path = OUTPUT_DIRECTORY / 'ocr_index.jsonl'
with index_path.open('w', encoding='utf-8') as handle:
    for row in RESULTS:
        mapping = load_keyframe_map(row['video_id'])
        mapped = mapping.get(row['keyframe'], {})
        payload = {
            'video_id': row['video_id'],
            'keyframe': Path(row['path']).name,
            'frame_idx': mapped.get('frame_idx'),
            'pts_time': mapped.get('pts_time'),
            'fps': mapped.get('fps'),
            'ocr_text': row['text'],
            'ocr_text_normalized': clean_text(row['text']).casefold(),
            'ocr_boxes': row['boxes'],
            'sharpness': row['sharpness'],
            'model': f'{DETECTOR_MODEL}+{VIETOCR_CONFIG_NAME}',
        }
        handle.write(json.dumps(payload, ensure_ascii=False) + '\n')

summary = {
    'model': f'{DETECTOR_MODEL}+{VIETOCR_CONFIG_NAME}',
    'detector_score_floor': DETECTOR_SCORE_FLOOR,
    'recognizer_score_floor': RECOGNIZER_SCORE_FLOOR,
    'videos': sorted({row['video_id'] for row in RESULTS}),
    'frames': len(RESULTS),
    'frames_with_text': sum(bool(row['text']) for row in RESULTS),
    'frames_with_vietnamese_diacritic': sum(row['has_vietnamese_diacritic'] for row in RESULTS),
    'temporal_consensus': CONSENSUS_BY_VIDEO,
}
(OUTPUT_DIRECTORY / 'summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

import shutil
archive_path = shutil.make_archive('/content/aic_ocr_hybrid_test', 'zip', OUTPUT_DIRECTORY)
print('JSONL:', index_path)
print('Summary:', OUTPUT_DIRECTORY / 'summary.json')
print('ZIP:', archive_path)
print('\nNếu kết quả ổn, tải ZIP về hoặc chép OUTPUT_DIRECTORY sang Drive rồi mới mở rộng số video.')

## Cách đọc kết quả

- Box đúng nhưng text sai: detector ổn, cần chỉnh crop/padding hoặc recognizer.
- Không có box trên chữ nhìn thấy: giảm `DETECTOR_SCORE_FLOOR` xuống `0.20` hoặc thử `PP-OCRv6_medium_det`.
- Text mất dấu: tăng `pad_y` trong `rectify_crop()` từ `0.35` lên `0.50`, sau đó thử `vgg_transformer`.
- Text đúng ở một frame nhưng sai ở frame kế bên: giữ kết quả có confidence/sharpness tốt nhất và dùng `temporal_consensus`.
- Nhiều box rác: tăng `DETECTOR_SCORE_FLOOR` hoặc `MAX_BOXES_PER_IMAGE`; không hạ ngưỡng recognizer để “cứu” box sai.

Chỉ sau khi smoke test nhìn ổn mới chuyển logic này vào worker chạy toàn dataset. Khi đó nên giữ cả
box dưới ngưỡng trong JSON để có thể đổi threshold mà không chạy lại GPU.